In [1]:
import ase
import ase.io
import numpy as np
import pyscf
import time
import os
from pyscf import gto, dft, df, lib
from pyscf.scf import hf
import scipy
import equiv_dens.utils.base as utils
hf.MUTE_CHKFILE = True
%load_ext autoreload
%autoreload 2

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


Use "numpy" for Fourier Transform


In [2]:
# load data
load_path = 'datasets/thiophene1mer_Bidx-100.xyz'
mols = list(ase.io.iread(load_path))
basis = 'augccpvdz'
auxbasis = 'augccpvqzjkfit'

In [ ]:

save_path = load_path.split('.')[0] + '_pyscf_' + basis + '.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
for i in range(len(results), len(mols)):
    print('calc', i)
    start = time.time()
    pos = mols[i].get_positions()
    atom_nums = mols[i].get_atomic_numbers()
    atom = []
    for j in range(len(atom_nums)):
        atom.append((atom_nums[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis=basis)
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    #mf.max_cycle = 1000
    mf.kernel()
    g = mf.nuc_grad_method()
    gradients = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    print('mo occ', mf.mo_occ)
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = -gradients/ase.units.Bohr 
    
    dm1 = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)
    auxmol = df.addons.make_auxmol(mol, auxbasis)

    ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
    ints_2c2e = auxmol.intor('int2c2e')
    print('ints3c2e shape', ints_3c2e.shape)
    print('ints2c2e shape', ints_2c2e.shape)
    
    nao = mol.nao
    naux = auxmol.nao
    df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
    df_coef = df_coef.reshape(naux, nao, nao)
    if dm1.ndim > 2:
        df_basis = []
        for j in range(dm1.shape[0]):
            df_basis.append(lib.einsum('Pij,ij->P', df_coef, dm1[j]))
        df_basis = np.stack(df_basis, axis=0)
        print(df_basis.shape)
            
    else:
        df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

    calc_dict['df_coeff'] = df_basis
    calc_dict['auxbasis'] = auxbasis
    res.append(calc_dict)
    
    if i%10 == 0:
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)

In [3]:
# load data
pyscf_path = 'datasets/thiophene1mer_Bidx-100_pyscf_augccpvdz.npy'
load_path = 'datasets/thiophene1mer_Bidx-100.xyz'
mols = list(ase.io.iread(load_path))
atom_pos = utils.ase_to_npy(mols)
atom_types = []
atom_numbers = []
for i in range(len(mols)):
    atom_types.append(mols[i].get_chemical_symbols())
    atom_numbers.append(mols[i].get_atomic_numbers())

print(atom_pos.shape)
npy_path = 'datasets/thiophene1mer_Bidx-100.npy'
pyscf_data = np.load(pyscf_path, allow_pickle=True)
len_pyscf = len(pyscf_data)
print('pyscf data len', len_pyscf)
data = {}
data['positions'] = atom_pos[:len_pyscf]
data['atom_numbers'] = np.array(atom_numbers)[:len_pyscf]
data['atom_types'] = atom_types[:len_pyscf]
data['energy'] = []
data['forces'] = []
for calc in pyscf_data:
    data['energy'].append(calc[1]['energy'])
    data['forces'].append(calc[1]['forces'])
    
data['energy'] = np.array(data['energy'])[:, None]
data['forces'] = np.array(data['forces'])
for key in data.keys():
    if isinstance(data[key], np.ndarray):
        print(key, 'shape', data[key].shape)
    else:
        print(key, 'length', len(data[key]))
np.save(npy_path, data, allow_pickle=True)

(100, 9, 3)
pyscf data len 100
positions shape (100, 9, 3)
atom_numbers shape (100, 9)
atom_types length 100
energy shape (100, 1)
forces shape (100, 9, 3)


In [3]:
print(pyscf_data[20][0]['atom'])
print(data['positions'][20])

NameError: name 'pyscf_data' is not defined

In [17]:
print(data['positions'].shape[0])
print(data['atom_numbers'].shape[0])
mols = utils.npy_to_ase(data['positions'], data['atom_numbers'][0, :])
print(mols)

61
61
[Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3', pbc=False), Atoms(symbols='HSC4H3

In [18]:
from equiv_dens.utils.grids import spherical_grid

new_data = np.copy(data).item()
new_data['atom_numbers'] = data['atom_numbers'][0]
new_data['atom_types'] = data['atom_types'][0]

print(spherical_grid(new_data, level=1))

len atom types 9
atom numbers 1
level 1
{'H': (tensor([[ 8.2905e-05,  0.0000e+00,  0.0000e+00],
        [ 7.6167e-04,  0.0000e+00,  0.0000e+00],
        [ 2.7866e-03,  0.0000e+00,  0.0000e+00],
        ...,
        [ 0.0000e+00, -8.9777e-01, -4.8905e-01],
        [ 0.0000e+00, -1.0629e+00, -5.7901e-01],
        [ 0.0000e+00, -1.2507e+00, -6.8130e-01]], dtype=torch.float64), tensor([1.9635e-12, 7.6118e-10, 2.4844e-08,  ..., 1.5140e-01, 2.4135e-01,
        3.8005e-01], dtype=torch.float64)), 'S': (tensor([[ 1.6855e-05,  0.0000e+00,  0.0000e+00],
        [ 1.5488e-04,  0.0000e+00,  0.0000e+00],
        [ 5.6678e-04,  0.0000e+00,  0.0000e+00],
        ...,
        [-1.0372e+00, -1.6513e+00, -3.1414e-01],
        [-1.1378e+00, -1.8115e+00, -3.4462e-01],
        [-1.2479e+00, -1.9868e+00, -3.7795e-01]], dtype=torch.float64), tensor([1.6501e-14, 6.4006e-12, 2.0910e-10,  ..., 3.3565e-01, 4.4085e-01,
        5.8139e-01], dtype=torch.float64)), 'C': (tensor([[ 3.3888e-05,  0.0000e+00,  0.0000e+0

In [19]:
from equiv_dens.utils.grids import spherical_grid2


print(spherical_grid2(data, level=1))

level 1
{'H': (tensor([[ 8.2905e-05,  0.0000e+00,  0.0000e+00],
        [ 7.6167e-04,  0.0000e+00,  0.0000e+00],
        [ 2.7866e-03,  0.0000e+00,  0.0000e+00],
        ...,
        [ 0.0000e+00, -8.9777e-01, -4.8905e-01],
        [ 0.0000e+00, -1.0629e+00, -5.7901e-01],
        [ 0.0000e+00, -1.2507e+00, -6.8130e-01]], dtype=torch.float64), tensor([1.9635e-12, 7.6118e-10, 2.4844e-08,  ..., 1.5140e-01, 2.4135e-01,
        3.8005e-01], dtype=torch.float64)), 'C': (tensor([[ 3.3888e-05,  0.0000e+00,  0.0000e+00],
        [ 3.1137e-04,  0.0000e+00,  0.0000e+00],
        [ 1.1394e-03,  0.0000e+00,  0.0000e+00],
        ...,
        [-9.7328e-01, -1.5496e+00, -2.9478e-01],
        [-1.0926e+00, -1.7395e+00, -3.3090e-01],
        [-1.2256e+00, -1.9513e+00, -3.7120e-01]], dtype=torch.float64), tensor([1.3410e-13, 5.2007e-11, 1.6985e-09,  ..., 3.4703e-01, 4.8633e-01,
        6.8489e-01], dtype=torch.float64)), 'S': (tensor([[ 1.6855e-05,  0.0000e+00,  0.0000e+00],
        [ 1.5488e-04,  0.000